# S3·MIÉ — Un motor de grafo de estados en 40 líneas

**MMIA 6013 · Semana 3 · Miércoles**

Construyes desde cero la abstracción central de LangGraph: estado + nodos +
aristas condicionales + checkpoints. Entender el motor por dentro hace que
LangGraph deje de ser magia.

Todo local, sin dependencias. Al final, el mismo grafo en LangGraph (opcional).

In [ ]:
import json

## Parte 1 — El motor

### ✏️ Ejercicio 1.1
Completa la clase `Grafo`:
- `agregar_nodo(nombre, fn)`: fn recibe el estado (dict) y devuelve el estado modificado
- `agregar_arista(desde, hasta)`: transición fija
- `agregar_arista_condicional(desde, router)`: router(estado) → nombre del siguiente nodo
- `ejecutar(estado, inicio, max_pasos=20)`: corre hasta llegar a "FIN",
  registrando en `estado["_traza"]` cada nodo visitado

In [ ]:
# Ejercicio 1.1 — un LangGraph de bolsillo

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

## Parte 2 — El grafo del agente: clasificar → (SQL | RAG) → redactar

Los nodos son simulaciones deterministas (en tu Lab serían llamadas al LLM y
a tus tools reales). Observa que cada nodo SOLO transforma el estado.

In [ ]:
def nodo_clasificar(estado):
    p = estado["pregunta"].lower()
    if any(w in p for w in ("ventas", "total", "cuánto", "promedio")):
        estado["ruta"] = "sql"
    elif any(w in p for w in ("política", "vacaciones", "remoto", "documento")):
        estado["ruta"] = "rag"
    else:
        estado["ruta"] = "sensible"     # lo desconocido pasa por un humano
    return estado

def nodo_sql(estado):
    estado["datos"] = {"total_marzo": 2695, "n_ventas": 4}   # simulación
    return estado

def nodo_rag(estado):
    estado["datos"] = {"chunk": "El trabajo remoto está permitido hasta 3 días por semana."}
    return estado

def nodo_aprobacion(estado):
    if not estado.get("aprobado"):
        estado["_pausa"] = True          # checkpoint: esperar al humano
    return estado

def nodo_redactar(estado):
    estado["respuesta"] = f"[respuesta redactada con {json.dumps(estado.get('datos', {}), ensure_ascii=False)}]"
    return estado

### ✏️ Ejercicio 2.1
Arma el grafo con los nodos y las aristas de la parte anterior, y córrelo.

Lo que hay que ver es **de qué habla un grafo que un loop no puede decir**: el estado
viaja explícito entre nodos, y cada arista es una decisión que se puede leer sin ejecutar
nada. Un `while` con `if`s dispersos hace lo mismo y no se puede dibujar.

In [ ]:
# Ejercicio 2.1 — arma el grafo:
#   clasificar --(ruta=sql)--> sql --> redactar --> FIN
#              --(ruta=rag)--> rag --> redactar --> FIN
#              --(ruta=sensible)--> aprobacion --> redactar --> FIN

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

## Parte 3 — Human-in-the-loop con checkpoint

Una pregunta "sensible" llega al nodo de aprobación, que **pausa** el grafo.
El estado pausado es serializable (JSON): puede guardarse en disco/DB y
reanudarse horas después — exactamente lo que hace el checkpointer de LangGraph.

### ✏️ Ejercicio 3.1
Ejecuta una pregunta sensible, observa la pausa del checkpoint, y reanuda el grafo con la
decisión humana —una aprobando, otra rechazando—. Anota en qué nodo se detuvo y qué parte
del estado sobrevivió a la pausa.

Lo que hay que notar: el estado se **serializa** en la pausa. Un agente que solo vive en
variables de Python no se puede pausar para esperar a una persona, y esperar a una persona
es el caso normal en cuanto una acción es irreversible — el freno del jueves.

In [ ]:
# Ejercicio 3.1 — ejecuta una pregunta sensible, observa la pausa, "aprueba"
# y reanuda desde el checkpoint.

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

## Parte 4 (opcional) — El mismo grafo en LangGraph real

Si tienes `pip install langgraph==1.2.12`, la versión que fija el curso, esto es la traducción directa.
Compara: misma semántica, más baterías (checkpointer persistente, streaming,
reintentos, visualización).

In [ ]:
try:
    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END

    class Estado(TypedDict, total=False):
        pregunta: str
        ruta: str
        datos: dict
        respuesta: str

    sg = StateGraph(Estado)
    sg.add_node("clasificar", nodo_clasificar)
    sg.add_node("sql", nodo_sql)
    sg.add_node("rag", nodo_rag)
    sg.add_node("redactar", nodo_redactar)
    sg.add_edge(START, "clasificar")
    sg.add_conditional_edges("clasificar",
        lambda e: e["ruta"] if e["ruta"] in ("sql", "rag") else "sql",
        {"sql": "sql", "rag": "rag"})
    sg.add_edge("sql", "redactar")
    sg.add_edge("rag", "redactar")
    sg.add_edge("redactar", END)

    app = sg.compile()
    print(app.invoke({"pregunta": "¿Cuánto vendimos en marzo?"})["respuesta"])
except ImportError:
    print("langgraph no instalado — sección de lectura. `pip install langgraph==1.2.12` para probarla.")

## Parte 5 — De tools locales a un servidor MCP

El lunes escribiste tres tools y las cableaste a mano dentro de tu loop. Eso
funciona para UNA app. El problema aparece al escalar:

- **M** aplicaciones (tu agente, el IDE, el chat de soporte, un cron)
- **N** herramientas (ventas, RAG, calendario, Jira)
- Sin estándar: **M × N** integraciones, cada una a mano.

MCP (Model Context Protocol) es un protocolo cliente-servidor para ese
problema: cada herramienta se expone UNA vez como servidor, cada aplicación
habla el mismo protocolo como cliente. **M + N** en vez de M × N.

Aquí no instalamos nada: simulamos el servidor en el propio notebook, igual
que el lunes simulamos el LLM. Lo que importa es la **forma** de los mensajes.

In [ ]:
# Un servidor MCP mínimo: publica un catálogo y ejecuta llamadas por nombre.
# Son las MISMAS tres tools del lunes, ahora detrás de un protocolo.
VENTAS_MES = {"2026-01": 41200, "2026-02": 38750, "2026-03": 52300}
PRODUCTOS = ["licencia", "soporte", "capacitación"]

def _consultar_ventas(mes):
    return {"mes": mes, "total": VENTAS_MES.get(mes, 0)}

def _listar_productos():
    return {"productos": PRODUCTOS}

class ServidorMCP:
    """Servidor MCP simulado. Solo dos métodos: tools/list y tools/call."""

    def __init__(self, nombre):
        self.nombre = nombre
        self._impl = {}
        self._catalogo = []

    def publicar(self, name, description, input_schema, fn):
        self._catalogo.append({"name": name, "description": description,
                               "inputSchema": input_schema})
        self._impl[name] = fn

    def manejar(self, metodo, params=None):
        params = params or {}
        if metodo == "tools/list":
            return {"tools": self._catalogo}
        if metodo == "tools/call":
            nombre = params["name"]
            if nombre not in self._impl:
                return {"isError": True, "content": f"tool desconocida: {nombre}"}
            return {"isError": False,
                    "content": self._impl[nombre](**params.get("arguments", {}))}
        return {"isError": True, "content": f"método no soportado: {metodo}"}

servidor = ServidorMCP("ventas-usfq")
servidor.publicar(
    "consultar_ventas",
    "Devuelve el total vendido en un mes dado (formato YYYY-MM).",
    {"type": "object",
     "properties": {"mes": {"type": "string", "description": "Mes YYYY-MM"}},
     "required": ["mes"]},
    _consultar_ventas)
servidor.publicar(
    "listar_productos",
    "Lista los productos distintos que aparecen en las ventas.",
    {"type": "object", "properties": {}},
    _listar_productos)

print(json.dumps(servidor.manejar("tools/list"), ensure_ascii=False, indent=2))

### ✏️ Ejercicio 5.1 — el cliente

Escribe `ClienteMCP`, que es lo que tu agente usaría en lugar del diccionario
`TOOLS` cableado del lunes:

- `descubrir()`: pide `tools/list` y guarda el catálogo. **El cliente no sabe
  de antemano qué herramientas hay** — esa es la diferencia con el lunes.
- `esquemas_para_llm()`: devuelve el catálogo en el formato que espera la API
  del modelo (`name`, `description`, `input_schema`).
- `invocar(nombre, argumentos)`: hace `tools/call` y devuelve el contenido.

In [ ]:
# Ejercicio 5.1

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

### ✏️ Ejercicio 5.2 — la cuenta que justifica el estándar

Un segundo servidor entra en juego y el agente no cambia ni una línea: solo
descubre más herramientas. Eso es el desacople.

In [ ]:
# Ejercicio 5.2

# ✏️ TU CÓDIGO AQUÍ
# Completa esta celda siguiendo las instrucciones anteriores.

In [ ]:
# 🤔 La cuenta, con tus números (comentario):
# a) En tu organización: ¿cuántas aplicaciones (M) usarían herramientas de IA?
#    ¿Cuántas herramientas/sistemas (N) querrías exponer?
# b) M × N = ____ integraciones a mano.  M + N = ____ con protocolo.
# c) ¿Qué NO resuelve MCP? (pista: piensa en permisos, en quién ejecuta,
#    y en el problema de la sesión de mañana)
#
# a)
# b)
# c)

In [ ]:
# 🤔 Decisión de arquitectura (comentario):
# Para tu Lab 03: ¿loop simple o grafo? Dibuja tu flujo en texto (nodos y
# flechas) y justifica la elección en 2-3 frases.
#
# Flujo:
#
# Justificación:
#

## Cierre

- Grafo de estados = nodos (funciones) + aristas condicionales + estado compartido.
- Checkpoint = estado serializable + punto de reanudación → human-in-the-loop limpio.
- LangGraph te da esto con baterías; ahora sabes qué hay debajo.
- MCP: un servidor publica `tools/list`, un cliente invoca `tools/call`. El agente
  descubre herramientas en vez de tenerlas cableadas → M + N en vez de M × N.
- MCP no reemplaza a function calling: function calling es cómo el modelo PIDE,
  MCP es cómo la herramienta se PUBLICA. Y ejecutar sigue siendo cosa de tu código.

**Mañana:** todo lo que puede salir mal — riesgos, costos descontrolados,
inyección vía tools — y la ética de los agentes autónomos.